# Task 1 (6p)
Your task is to modify the custom implementation of MultiHeadAttention. This custom implementation, currently, enables each token to attent to every other token.


Your job is to change this behavior in a specific way.
Let $S$ be our input sequence of length $2 \cdot k$:
- tokens on positions $i \lt k$ should attend to prefix of $S$ of length $k$ ($S[:k]$) - every token up to position k
- tokens on positions $i \ge k$ should attend to prefix of $S$  of length $i + 1$ ($S[:i + 1]$) - every previous token and itself

(Note: You can assume the sequence length is always an even number).

In [17]:
import torch
import math
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_head):
      super().__init__()
      self.d_model = d_model
      self.num_heads = num_heads
      self.d_head = d_head

      self.W_Q = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_K = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_V = torch.nn.Linear(d_model, num_heads*d_head, bias=True)
      self.W_O = torch.nn.Linear(num_heads*d_head, d_model, bias=True)

    def forward(self, x):

      seq_len, batch_size, _ = x.shape

      Q = self.W_Q(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      K = self.W_K(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)
      V = self.W_V(x).reshape(seq_len, batch_size, self.num_heads, self.d_head)

      scaled_QK = torch.einsum("ibhd,jbhd->bhij", Q, K) / math.sqrt(self.d_head)
      # shape of scaled_QK is (batch_size, num_heads, seq_len, seq_len)

      #TODO
      k = seq_len // 2
      row_indices = torch.arange(seq_len).unsqueeze(1)
      col_indices = torch.arange(seq_len).unsqueeze(0)

      mask = torch.full((seq_len, seq_len), -torch.inf)
      condition = (((row_indices >= k) & (row_indices >= col_indices)) | ((row_indices < k) & (col_indices < k)))
      mask[condition] = 0

      scaled_QK = scaled_QK + mask.view(1, 1, seq_len, seq_len)                                          

      #ENDTODO

      weights = F.softmax(scaled_QK, -1)
      attention = torch.einsum("bhij,jbhd->ibhd", weights, V)

      result = self.W_O(attention.reshape(seq_len, batch_size,self.num_heads * self.d_head))

      return result, weights

In [18]:
# Test your solution
d_model = 10
num_heads= 4
d_head = 5
k = 10
batch_size = 16

mha = MultiHeadAttention(d_model, num_heads, d_head)
batched_x= torch.randn((2*k, batch_size, d_model))
with torch.no_grad():
  result, weights = mha(batched_x)
print("Result:", result)
print("Weights:", weights)

Result: tensor([[[-0.1721,  0.1041, -0.0821,  ..., -0.2490, -0.2391,  0.0461],
         [-0.1086,  0.1946, -0.0612,  ..., -0.4196, -0.0894,  0.1438],
         [ 0.0421,  0.0357, -0.1455,  ..., -0.3426, -0.2881,  0.0884],
         ...,
         [-0.1299,  0.1389, -0.1342,  ..., -0.4345, -0.1980,  0.1453],
         [-0.0283,  0.2207, -0.0189,  ..., -0.4096, -0.1284,  0.1340],
         [-0.1333,  0.4501, -0.0539,  ..., -0.2913, -0.2685,  0.2614]],

        [[-0.0737,  0.0807,  0.0188,  ..., -0.1489, -0.2764,  0.1146],
         [-0.1011,  0.1882, -0.0605,  ..., -0.3703, -0.1021,  0.1580],
         [ 0.0051,  0.0053, -0.1800,  ..., -0.3255, -0.2855,  0.1307],
         ...,
         [-0.1215,  0.1349, -0.1465,  ..., -0.4446, -0.2339,  0.1683],
         [-0.0064,  0.1955, -0.0158,  ..., -0.2826, -0.2019,  0.2119],
         [-0.1152,  0.5781,  0.0626,  ..., -0.3439, -0.1493,  0.1897]],

        [[-0.0921,  0.1567,  0.0081,  ..., -0.1856, -0.2131,  0.0548],
         [-0.0722,  0.2236, -0.0334, 